# Finite Discrete-Time Fully Observable Markov Decision Process (MDP)

In [97]:
import gymnasium as gym

class MDP_Blackjack:
    '''Blackjack environment from Gymnasium.'''
    
    def __init__(self, policy=None):
        self.env = gym.make("Blackjack-v1", natural=False, sab=False)
        self.policy = policy
        
        self.episode_over = False
        self.total_reward = 0
        self.t = 0

        self.history = []
        self.transitions = []

    # S: State space
    def states(self):
        return self.env.observation_space

    # A: Action space
    def actions(self):
        return {0: "stick", 1: "hit"}

    # D: Decision epochs
    def decision_point(self, t):
        self.t = t
        return self.t

    # T: Record an observed transition
    def transition(self, state_1, action, state_2, t):
        transition = (state_1, action, state_2, t)
        self.transitions.append(transition)

    # R: Immediate reward received from the environment
    def reward(self, reward):
         self.total_reward += reward
         return reward

    # Utility: Start an episode
    def get_current_state(self, seed=None):
        self.observation, self.info = self.env.reset(seed=seed)
        print(f"Current observation: {self.observation},",
              f"Current info: {self.info}")

        self.episode_over = False
        self.total_reward = 0
        self.t = 0
        self.history = []
        self.transitions = []

        return self.observation

    # Utility: Execute one action
    def take_action(self):
        state = self.observation
        t = self.decision_point(self.t)

        action = self.policy(state) # Policy
        next_state, reward, terminated, truncated, info = self.env.step(action)

        self.observation = next_state # Update the state
        self.reward(reward) # Assign a reward
        self.episode_over = terminated or truncated

        self.transition(state, action, next_state, t)
        self.history.append((state, action, reward, next_state))

        self.t += 1

        return next_state, reward

    # Policy: Random
    def random(self, state):
        action = self.env.action_space.sample() 
        print(f"Action: {self.actions()[action]}")
        return action

    def markovian(self, state):
        raise NotImplementedError("Policy not implemented yet.")

    def stationary(self, state):
        raise NotImplementedError("Policy not implemented yet.")

    def prob_history(self, state, history):
        raise NotImplementedError("Policy not implemented yet.")

    def deterministic_history(self, state, history):
        raise NotImplementedError("Policy not implemented yet.")

    def close(self):
        self.env.close()

In [105]:
def run_episode(mdp, seed=None):
    mdp.get_current_state(seed=seed)

    while not mdp.episode_over:
        state, reward = mdp.take_action()
        print(f"State: {state}, Reward: {reward}\n") 

    print("Total reward:", mdp.total_reward)

    return mdp.history


mdp = MDP_Blackjack()
mdp.policy = mdp.random
history = run_episode(mdp, seed=42)
mdp.close()

Current observation: (15, 2, 0), Current info: {}
Action: stick
State: (15, 2, 0), Reward: 1.0

Total reward: 1.0
